<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Lab: Exploratory Data Analysis**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform Exploratory Data Analysis or EDA. 


## Objectives


In this lab, you will perform the following:


- Examine the structure of a dataset.

- Handle missing values effectively.

- Conduct summary statistics on key columns.

- Analyze employment status, job satisfaction, programming language usage, and trends in remote work.


## Hands on Lab


#### Step 1: Install and Import Libraries


Install the necessary libraries for data manipulation and visualization.


In [ ]:
!pip install pandas
!pip install matplotlib
!pip install seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#### Step 2: Load and Preview the Dataset
Load the dataset from the provided URL. Use df.head() to display the first few rows to get an overview of the structure.


In [ ]:
# Load the Stack Overflow survey dataset
data_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv'
df = pd.read_csv(data_url)

# Set pandas option to display all columns
pd.set_option('display.max_columns', None)

# Display the first few rows of the dataset
df.head()

#### Step 3: Handling Missing Data


Identify and manage missing values in critical columns such as `Employment`, `JobSat`, and `RemoteWork`. Implement a strategy to fill or drop these values, depending on the significance of the missing data.


In [ ]:
# Step 3: Handle missing data in Employment, JobSat and RemoteWork
key_cols = ['Employment', 'JobSat', 'RemoteWork']
print("Missing values before:")
print(df[key_cols].isnull().sum())

# JobSat is a 0-10 score, so fill it with the median; the others are categories, so use the most frequent value
df['JobSat'] = pd.to_numeric(df['JobSat'], errors='coerce')
df['JobSat'] = df['JobSat'].fillna(df['JobSat'].median())
df['Employment'] = df['Employment'].fillna(df['Employment'].mode()[0])
df['RemoteWork'] = df['RemoteWork'].fillna(df['RemoteWork'].mode()[0])

print("\nMissing values after:")
print(df[key_cols].isnull().sum())

#### Step 4: Analysis of Experience and Job Satisfaction


Analyze the relationship between years of professional coding experience (`YearsCodePro`) and job satisfaction (`JobSat`). Summarize `YearsCodePro` and calculate median satisfaction scores based on experience ranges.

- Create experience ranges for `YearsCodePro` (e.g., `0-5`, `5-10`, `10-20`, `>20` years).

- Calculate the median `JobSat` for each range.

- Visualize the relationship using a bar plot or similar visualization.


In [ ]:
# Step 4: Experience ranges vs median job satisfaction
df['YearsCodePro_Num'] = pd.to_numeric(
    df['YearsCodePro'].replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
    errors='coerce')
print(df['YearsCodePro_Num'].describe())

df['ExperienceRange'] = pd.cut(df['YearsCodePro_Num'], bins=[-1, 5, 10, 20, 100],
                               labels=['0-5', '5-10', '10-20', '>20'])

median_jobsat = df.groupby('ExperienceRange', observed=True)['JobSat'].median()
print("\nMedian JobSat by experience range:")
print(median_jobsat)

median_jobsat.plot(kind='bar', color='steelblue', figsize=(8, 5))
plt.title('Median Job Satisfaction by Years of Professional Coding')
plt.xlabel('Years of professional coding experience')
plt.ylabel('Median JobSat (0-10)')
plt.xticks(rotation=0)
plt.show()

#### Step 5: Visualize Job Satisfaction


Use a count plot to show the distribution of `JobSat` values. This provides insights into the overall satisfaction levels of respondents.


In [ ]:
# Step 5: Count plot of JobSat
plt.figure(figsize=(10, 5))
sns.countplot(x='JobSat', data=df, color='steelblue')
plt.title('Distribution of Job Satisfaction (JobSat)')
plt.xlabel('Job satisfaction score (0 = lowest, 10 = highest)')
plt.ylabel('Number of respondents')
plt.show()

#### Step 6: Analyzing Remote Work Preferences by Job Role


Analyze trends in remote work based on job roles. Use the `RemoteWork` and `Employment` columns to explore preferences and examine if specific job roles prefer remote work more than others.

- Use a count plot to show remote work distribution.

- Cross-tabulate remote work preferences by employment type (e.g., full-time, part-time) and job roles.


In [ ]:
# Step 6: Remote work distribution and preferences by employment type and job role
plt.figure(figsize=(8, 5))
sns.countplot(y='RemoteWork', data=df, order=df['RemoteWork'].value_counts().index, color='steelblue')
plt.title('Remote Work Distribution')
plt.xlabel('Number of respondents')
plt.ylabel('')
plt.show()

# Cross-tab by employment type (top 5 employment answers), shown as row percentages
top_emp = df['Employment'].value_counts().head(5).index
emp_ct = pd.crosstab(df[df['Employment'].isin(top_emp)]['Employment'],
                     df[df['Employment'].isin(top_emp)]['RemoteWork'], normalize='index') * 100
print("Remote work by employment type (% of each row):")
print(emp_ct.round(1))

# Cross-tab by job role (DevType, top 10 roles)
top_roles = df['DevType'].value_counts().head(10).index
roles = df[df['DevType'].isin(top_roles)]
role_ct = pd.crosstab(roles['DevType'], roles['RemoteWork'], normalize='index') * 100
print("\nRemote work by job role (% of each row):")
print(role_ct.round(1))

role_ct.plot(kind='barh', stacked=True, figsize=(12, 6))
plt.title('Remote Work Preference by Job Role (top 10 roles)')
plt.xlabel('% of respondents')
plt.legend(title='RemoteWork', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Step 7: Analyzing Programming Language Trends by Region


Analyze the popularity of programming languages by region. Use the `LanguageHaveWorkedWith` column to investigate which languages are most used in different regions.

- Filter data by country or region.

- Visualize the top programming languages by region with a bar plot or heatmap.


In [ ]:
# Step 7: Top programming languages by region (top 5 countries)
top_countries = df['Country'].value_counts().head(5).index
lang = df[df['Country'].isin(top_countries)][['Country', 'LanguageHaveWorkedWith']].dropna()

# Each answer lists several languages separated by ';' -> one row per language
lang = lang.assign(Language=lang['LanguageHaveWorkedWith'].str.split(';')).explode('Language').reset_index(drop=True)

# Share of respondents in each country who use each language
respondents = df[df['Country'].isin(top_countries)].groupby('Country')['LanguageHaveWorkedWith'].count()
usage = pd.crosstab(lang['Language'], lang['Country']).div(respondents, axis=1) * 100

top_langs = usage.mean(axis=1).sort_values(ascending=False).head(10).index
plt.figure(figsize=(10, 7))
sns.heatmap(usage.loc[top_langs], annot=True, fmt='.0f', cmap='Blues')
plt.title('Top 10 Languages by Country (% of respondents who use them)')
plt.xlabel('Country')
plt.ylabel('Language')
plt.show()

#### Step 8: Correlation Between Experience and Satisfaction


Examine how years of experience (`YearsCodePro`) correlate with job satisfaction (`JobSatPoints_1`). Use a scatter plot to visualize this relationship.


In [ ]:
# Step 8: Experience (YearsCodePro) vs job satisfaction (JobSatPoints_1)
df['YearsCodePro_Num'] = pd.to_numeric(
    df['YearsCodePro'].replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
    errors='coerce')
df['JobSatPoints_1'] = pd.to_numeric(df['JobSatPoints_1'], errors='coerce')
subset = df[['YearsCodePro_Num', 'JobSatPoints_1']].dropna()

print("Pearson correlation:", round(subset['YearsCodePro_Num'].corr(subset['JobSatPoints_1']), 3))

plt.figure(figsize=(10, 6))
plt.scatter(subset['YearsCodePro_Num'], subset['JobSatPoints_1'], alpha=0.3)
plt.title('Years of Professional Coding vs Job Satisfaction Points')
plt.xlabel('YearsCodePro')
plt.ylabel('JobSatPoints_1')
plt.show()

#### Step 9: Educational Background and Employment Type


Explore how educational background (`EdLevel`) relates to employment type (`Employment`). Use cross-tabulation and visualizations to understand if higher education correlates with specific employment types.


In [ ]:
# Step 9: Education level vs employment type
top_emp = df['Employment'].value_counts().head(5).index
edu = df[df['Employment'].isin(top_emp)]

edu_ct = pd.crosstab(edu['EdLevel'], edu['Employment'])
print(edu_ct)

# Row percentages make education groups of different sizes comparable
(pd.crosstab(edu['EdLevel'], edu['Employment'], normalize='index') * 100).plot(
    kind='barh', stacked=True, figsize=(12, 6))
plt.title('Employment Type by Education Level')
plt.xlabel('% of respondents')
plt.ylabel('Education level')
plt.legend(title='Employment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Step 10: Save the Cleaned and Analyzed Dataset


After your analysis, save the modified dataset for further use or sharing.


In [ ]:
# Step 10: Save the cleaned and analyzed dataset
df.to_csv('survey_data_cleaned_analyzed.csv', index=False)
print("Saved survey_data_cleaned_analyzed.csv with", df.shape[0], "rows and", df.shape[1], "columns")

<h2>Summary</h2>


In this revised lab, you:

- Loaded and explored the structure of the dataset.

- Handled missing data effectively.

- Analyzed key variables, including working hours, job satisfaction, and remote work trends.

- Investigated programming language usage by region and examined the relationship between experience and satisfaction.

- Used cross-tabulation to understand educational background and employment type.


Copyright © IBM Corporation. All rights reserved.
